In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer
from evedesign.utils import ensure_sequence

In [ ]:
seq = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
s = System([Protein(rep=seq, id='EcCM', first_index=2)])
inst = s.rep_to_instance()

In [ ]:
from evedesign.tools.mmseqs2 import add_sequences_mmseqs2

s = add_sequences_mmseqs2(s, use_pairing=True)

m = BoltzFoldTransformer(device='cpu', use_msa=True, diffusion_samples=1)
m.build(s)

In [ ]:
output_structures = m.transform([inst])

In [ ]:
result = output_structures[0]
print(f"Score (boltz2 confidence score): {result.score}")

In [ ]:
print("Confidence scores:")
for key, value in result.metadata.items():
    print(f"  {key}: {value}")
ei = result[0]
structures = ensure_sequence(ei.models["model_0"])

if ei.models:
    chain_id = structures[0].chains()[0]
    structure = structures[0]              
    print(f"\nChain: {chain_id}")
    print(f"Atom count: {len(structure.atom_array)}")
    print(f"Residue range: {structure.atom_array.res_id.min()} - {structure.atom_array.res_id.max()}")
    print(structure.atom_df().head(5))

In [ ]:
! pip install py3Dmol -q

In [ ]:
import io
import py3Dmol

ei = result[0]
if ei.models:
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]

    buf = io.StringIO()
    structure.to_file(buf, format="cif")
    cif_content = buf.getvalue()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(cif_content, "cif")
    view.setStyle({
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90
            }
        }
    })
    view.zoomTo()
    view.show()